# Лабораторна робота: Аналіз індексів VHI для областей України
### Імпорт бібліотек
У цій комірці ми підключаємо всі необхідні інструменти для роботи з файлами, мережею та даними.

In [5]:
import os
import urllib.request
import pandas as pd
import glob
from datetime import datetime
from IPython.display import display


### Завдання 1: Завантаження даних VHI
Функція `download_vhi_data` завантажує файли з сервера NOAA. 
- Використовує `urllib`.
- Додає дату та час до назви файлу.
- Перевіряє, чи файл вже існує, щоб уникнути повторного завантаження.
- Пропускає `provinceID=0` (середнє по Україні).

In [2]:
def download_vhi_data():
    folder = "vhi_data"
    if not os.path.exists(folder):
        os.makedirs(folder)

    for i in range(1, 28):
        existing_files = glob.glob(os.path.join(folder, f"vhi_id_{i}_*.csv"))
        if existing_files:
            print(f"Дані для області {i} вже завантажені. Пропускаємо.")
            continue

        url = f"https://www.star.nesdis.noaa.gov/smcd/emb/vci/VH/get_TS_admin.php?country=UKR&provinceID={i}&year1=1981&year2=2024&type=Mean"
        
        try:
            with urllib.request.urlopen(url) as response:
                content = response.read().decode('utf-8')
            timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
            filename = os.path.join(folder, f"vhi_id_{i}_{timestamp}.csv")
            
            with open(filename, 'w') as f:
                f.write(content)
            print(f"Область {i}: успішно завантажено у {filename}")
        except Exception as e:
            print(f"Область {i}: помилка — {e}")


download_vhi_data()

Дані для області 1 вже завантажені. Пропускаємо.
Дані для області 2 вже завантажені. Пропускаємо.
Дані для області 3 вже завантажені. Пропускаємо.
Дані для області 4 вже завантажені. Пропускаємо.
Дані для області 5 вже завантажені. Пропускаємо.
Дані для області 6 вже завантажені. Пропускаємо.
Дані для області 7 вже завантажені. Пропускаємо.
Дані для області 8 вже завантажені. Пропускаємо.
Дані для області 9 вже завантажені. Пропускаємо.
Дані для області 10 вже завантажені. Пропускаємо.
Дані для області 11 вже завантажені. Пропускаємо.
Дані для області 12 вже завантажені. Пропускаємо.
Дані для області 13 вже завантажені. Пропускаємо.
Дані для області 14 вже завантажені. Пропускаємо.
Дані для області 15 вже завантажені. Пропускаємо.
Дані для області 16 вже завантажені. Пропускаємо.
Дані для області 17 вже завантажені. Пропускаємо.
Дані для області 18 вже завантажені. Пропускаємо.
Дані для області 19 вже завантажені. Пропускаємо.
Дані для області 20 вже завантажені. Пропускаємо.
Дані для 

### Завдання 2  
### Зчитування завантажених текстових файлів у pandas DataFrame та очищення даних

In [26]:
def load_vhi_data(data_dir="vhi_data"):
    frames = []

    for filename in os.listdir(data_dir):
        if not filename.endswith(".csv"):
            continue

        file_parts = filename.replace(".csv", "").split("_")

        province_id = None
        for part in file_parts:
            if part.isdigit():
                num = int(part)
                if 1 <= num <= 27:
                    province_id = num
                    break

        if province_id is None:
            print(f"Не вдалося визначити номер області у файлі: {filename}")
            continue

        filepath = os.path.join(data_dir, filename)

        try:
            temp_df = pd.read_csv(
                filepath,
                header=1,
                names=["year", "week", "SMN", "SMT", "VCI", "TCI", "VHI"],
                sep=",",
                engine="python",
                skipfooter=1,
                na_values=["", " ", "NULL", "NaN"],
            )
        except Exception as e:
            print(f"Помилка читання {filename}: {e}")
            continue

        # Прибираємо HTML-сміття, яке є у файлах NOAA
        temp_df["year"] = temp_df["year"].astype(str).str.replace("<tt><pre>", "", regex=False).str.strip()
        temp_df["VHI"] = temp_df["VHI"].astype(str).str.replace("</tt></pre>", "", regex=False).str.strip()

        numeric_cols = ["year", "week", "SMN", "SMT", "VCI", "TCI", "VHI"]
        temp_df[numeric_cols] = temp_df[numeric_cols].apply(pd.to_numeric, errors="coerce")

        # Видаляємо всі рядки, де хоча б в одному числовому стовпці є NaN
        temp_df = temp_df.dropna(subset=numeric_cols)

        temp_df = temp_df[temp_df["VHI"] != -1]

        if temp_df.empty:
            continue

        temp_df["year"] = temp_df["year"].astype(int)
        temp_df["week"] = temp_df["week"].astype(int)
        temp_df["province_id"] = province_id

        frames.append(temp_df)

    if not frames:
        return pd.DataFrame()

    return pd.concat(frames, ignore_index=True)


df = load_vhi_data()
if not df.empty:
    display(df.head(10))
else:
    print("Таблиця порожня. Дані не завантажено.")



,Рік,Тиждень,SMN,SMT,VCI,TCI,VHI,ID_NOAA,Назва_області,ID_області_укр
0,1982,1,0.038,252.22,47.05,69.15,58.10,8,Харківська,21
1,1982,2,0.036,253.39,45.39,63.52,54.45,8,Харківська,21
2,1982,3,0.033,254.22,41.22,60.22,50.72,8,Харківська,21
3,1982,4,0.031,255.60,37.35,58.01,47.68,8,Харківська,21
4,1982,5,0.030,257.16,32.54,56.92,44.73,8,Харківська,21
5,1982,6,0.028,258.12,25.63,57.18,41.41,8,Харківська,21
6,1982,7,0.029,259.90,21.30,54.76,38.03,8,Харківська,21
7,1982,8,0.033,262.82,21.84,49.76,35.80,8,Харківська,21
8,1982,9,0.036,265.27,20.87,48.45,34.66,8,Харківська,21
9,1982,10,0.041,267.96,20.64,48.64,34.64,8,Харківська,21


### Завдання 3
### Реалізація процедури зміни індексів областей

In [41]:
noaa_to_name_ua = {
    1: "Черкаська",
    2: "Чернігівська",
    3: "Чернівецька",
    4: "Крим",
    5: "Дніпропетровська",
    6: "Донецька",
    7: "Івано-Франківська",
    8: "Харківська",
    9: "Херсонська",
    10: "Хмельницька",
    11: "Київська",
    12: "м. Київ",
    13: "Кіровоградська",
    14: "Луганська",
    15: "Львівська",
    16: "Миколаївська",
    17: "Одеська",
    18: "Полтавська",
    19: "Рівненська",
    20: "Сумська",
    21: "Тернопільська",
    22: "Вінницька",
    23: "Волинська",
    24: "Закарпатська",
    25: "Запорізька",
    26: "Житомирська",
    27: "м. Севастополь"
}

ua_index = {
    "Вінницька": 1,
    "Волинська": 2,
    "Дніпропетровська": 3,
    "Донецька": 4,
    "Житомирська": 5,
    "Закарпатська": 6,
    "Запорізька": 7,
    "Івано-Франківська": 8,
    "Київська": 9,
    "м. Київ": 10,
    "Кіровоградська": 11,
    "Крим": 12,
    "Луганська": 13,
    "Львівська": 14,
    "Миколаївська": 15,
    "Одеська": 16,
    "Полтавська": 17,
    "Рівненська": 18,
    "Сумська": 19,
    "Тернопільська": 20,
    "Харківська": 21,
    "Херсонська": 22,
    "Хмельницька": 23,
    "Черкаська": 24,
    "Чернівецька": 25,
    "Чернігівська": 26,
    "м. Севастополь": 27
}
if not df.empty:
    df["Назва_області"] = df["ID_NOAA"].map(noaa_to_name_ua)
    df["ID_області_укр"] = df["Назва_області"].map(ua_index)

    df = df[df["ID_NOAA"].between(1, 27)].copy()
    df["ID_області_укр"] = df["ID_області_укр"].astype("Int64")

    df = df.rename(columns={
        "year": "Рік",
        "week": "Тиждень"
    })

    df = df.drop_duplicates(subset=["ID_області_укр", "Рік", "Тиждень", "VHI"]).copy()
    df.reset_index(drop=True, inplace=True)

    display(df.head(10))

    regions_table = (
        df[["ID_NOAA", "Назва_області", "ID_області_укр"]]
        .drop_duplicates()
        .sort_values("ID_NOAA")
        .reset_index(drop=True)
    )

    display(regions_table)

else:
    print("Таблиця порожня. Дані не завантажено.")

,Рік,Тиждень,SMN,SMT,VCI,TCI,VHI,ID_NOAA,Назва_області,ID_області_укр
0,1982,1,0.038,252.22,47.05,69.15,58.10,8,Харківська,21
1,1982,2,0.036,253.39,45.39,63.52,54.45,8,Харківська,21
2,1982,3,0.033,254.22,41.22,60.22,50.72,8,Харківська,21
3,1982,4,0.031,255.60,37.35,58.01,47.68,8,Харківська,21
4,1982,5,0.030,257.16,32.54,56.92,44.73,8,Харківська,21
5,1982,6,0.028,258.12,25.63,57.18,41.41,8,Харківська,21
6,1982,7,0.029,259.90,21.30,54.76,38.03,8,Харківська,21
7,1982,8,0.033,262.82,21.84,49.76,35.80,8,Харківська,21
8,1982,9,0.036,265.27,20.87,48.45,34.66,8,Харківська,21
9,1982,10,0.041,267.96,20.64,48.64,34.64,8,Харківська,21


,ID_NOAA,Назва_області,ID_області_укр
0,1,Черкаська,24
1,2,Чернігівська,26
2,3,Чернівецька,25
3,4,Крим,12
4,5,Дніпропетровська,3
5,6,Донецька,4
6,7,Івано-Франківська,8
7,8,Харківська,21
8,9,Херсонська,22
9,10,Хмельницька,23


## 4. Завдання

### 4.1. Реалізація процедури для формування вибірки: ряд VHI для області за вказаний рік

In [15]:
def get_vhi_series(df, province_id, year):
    result = df[(df["ID_області_укр"] == province_id) & (df["Рік"] == year)][["Тиждень", "VHI"]]
    return result

print("VHI для області 1 за 2020 рік")
display(get_vhi_series(df, province_id=1, year=2020).head())

VHI для області 1 за 2020 рік


,Тиждень,VHI
50018,1,41.75
50019,2,43.90
50020,3,45.12
50021,4,45.32
50022,5,44.78


### 4.2. Реалізація процедури для формування вибірки: ряд VHI за вказаний діапазон років для вказаних областей

In [19]:
def get_vhi_range(df, provinces, start_year, end_year):
    result = df[
        (df["ID_області_укр"].isin(provinces)) &
        (df["Рік"] >= start_year) &
        (df["Рік"] <= end_year)
    ][["ID_області_укр", "Назва_області", "Рік", "Тиждень", "VHI"]]

    result = result.drop_duplicates()

    return result.sort_values(["ID_області_укр", "Рік", "Тиждень"])
    print("VHI для областей 1, 5, 10 за 2018–2020 роки")
display(get_vhi_range(df, provinces=[1, 5, 10], start_year=2018, end_year=2020).head(20))

,ID_області_укр,Назва_області,Рік,Тиждень,VHI
49914,1,Вінницька,2018,1,53.05
49915,1,Вінницька,2018,2,55.41
49916,1,Вінницька,2018,3,57.44
49917,1,Вінницька,2018,4,56.72
49918,1,Вінницька,2018,5,54.35
49919,1,Вінницька,2018,6,52.21
49920,1,Вінницька,2018,7,49.09
49921,1,Вінницька,2018,8,48.81
49922,1,Вінницька,2018,9,49.79
49923,1,Вінницька,2018,10,50.07


### 4.3. Реалізація процедури пошуку екстремумів (min та max), середнього та медіани для вказаних областей та років

In [20]:
def get_vhi_stats(df, provinces, start_year, end_year):
    filtered = df[
        (df["ID_області_укр"].isin(provinces)) &
        (df["Рік"] >= start_year) &
        (df["Рік"] <= end_year)
    ]

    result = (
        filtered
        .groupby(["ID_області_укр", "Назва_області"])["VHI"]
        .agg(
            Мінімум="min",
            Максимум="max",
            Середнє="mean",
            Медіана="median"
        )
        .reset_index()
        .sort_values("ID_області_укр")
    )

    return result


print("Статистика VHI для областей 1, 5, 10 за 2018–2020 роки")
display(get_vhi_stats(df, provinces=[1, 5, 10], start_year=2018, end_year=2020))

Статистика VHI для областей 1, 5, 10 за 2018–2020 роки


,ID_області_укр,Назва_області,Мінімум,Максимум,Середнє,Медіана
0,1,Вінницька,19.45,70.74,49.013013,48.565
1,5,Житомирська,18.27,66.40,41.209872,41.285
2,10,м. Київ,24.95,60.83,43.476538,43.575
